In [1]:
import os
import mne
import torch
import numpy as np
from torch.utils.data import Dataset

class EEGMMIDB_Dataset_Optimized(Dataset):
    def __init__(self, data_dir, subjects, runs=[4, 6, 8, 10, 12, 14], tmin=0, tmax=4.0):
        self.data_dir = data_dir
        self.subjects = subjects
        self.runs = runs
        self.tmin = tmin
        self.tmax = tmax
        
        # 22 Channels explicitly centered around the Motor Cortex (C3, Cz, C4 region)
        self.target_channels = [
            'FC5.', 'FC3.', 'FC1.', 'FCz.', 'FC2.', 'FC4.', 'FC6.',
            'C5..', 'C3..', 'C1..', 'Cz..', 'C2..', 'C4..', 'C6..',
            'CP5.', 'CP3.', 'CP1.', 'CPz.', 'CP2.', 'CP4.', 'CP6.', 'Pz..'
        ]
        
        self.epochs = []
        self.labels = []
        self.subject_ids = []
        
        self.load_data()
        
    def load_data(self):
        mne.set_log_level('ERROR') # Hide verbose warnings
        
        for sub in self.subjects:
            sub_folder = f"S{sub:03d}"
            sub_path = os.path.join(self.data_dir, sub_folder)
            if not os.path.exists(sub_path):
                continue
                
            for run in self.runs:
                edf_file = os.path.join(sub_path, f"{sub_folder}R{run:02d}.edf")
                if not os.path.exists(edf_file):
                    continue
                    
                raw = mne.io.read_raw_edf(edf_file, preload=True)
                
                # --- Step 1: Bandpass Filter (7-30 Hz) & Notch Filter (50 Hz) ---
                raw.filter(l_freq=7.0, h_freq=30.0, fir_design='firwin', verbose=False)
                
                # --- Step 2: Pick Motor Cortex Specific Channels ---
                available_chs = [ch for ch in self.target_channels if ch in raw.ch_names]
                if len(available_chs) == 22:
                    raw.pick_channels(available_chs)
                else:
                    raw.pick(raw.ch_names[:22]) # Fallback
                
                raw.resample(250.0)
                
                events, event_id = mne.events_from_annotations(raw, verbose=False)
                
                mapping = {}
                if run in [4, 8, 12]:
                    mapping = {'T1': 0, 'T2': 1} # Left / Right fist
                elif run in [6, 10, 14]:
                    mapping = {'T1': 2, 'T2': 3} # Both fists / Both feet
                    
                if not mapping: continue
                
                try:
                    epochs = mne.Epochs(raw, events, event_id=mapping, tmin=self.tmin, 
                                        tmax=self.tmax-0.004, baseline=None, preload=True, verbose=False)
                    data = epochs.get_data() # (Trials, 22, 1000)
                    labels = epochs.events[:, -1]
                    
                    for i in range(len(data)):
                        self.epochs.append(data[i])
                        self.labels.append(labels[i])
                        self.subject_ids.append(sub - 1)
                except Exception:
                    continue

    def __len__(self):
        return len(self.epochs)

    def __getitem__(self, idx):
        x = torch.tensor(self.epochs[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        s = torch.tensor(self.subject_ids[idx], dtype=torch.long)
        
        # Robust Z-score Normalization
        mean = x.mean(dim=1, keepdim=True)
        std = x.std(dim=1, keepdim=True)
        x = (x - mean) / (std + 1e-6)
        
        return x, y, s

In [2]:
import torch.nn as nn
import torch.nn.functional as F
import math

class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_grl):
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        # Eq (13): dR/dZ = -lambda_grl * dL/dZ
        return grad_output.neg() * ctx.lambda_grl, None

def grl(x, lambda_grl=1.0):
    return GradientReversalLayer.apply(x, lambda_grl)

class SincFilterBank(nn.Module):
    """Learnable Sinc Filter Bank extracting F=10 frequency bands."""
    def __init__(self, in_channels=22, num_filters=10, kernel_size=81, sample_rate=250):
        super().__init__()
        self.num_filters = num_filters
        self.kernel_size = kernel_size
        self.sample_rate = sample_rate
        
        # Parameterize low (f1) and high (f2) cutoff frequencies
        self.f1 = nn.Parameter(torch.rand(num_filters) * 10 + 5) 
        self.f2 = nn.Parameter(torch.rand(num_filters) * 20 + 15)

    def forward(self, x):
        B, C, T = x.shape
        device = x.device
        
        n = torch.arange(-(self.kernel_size//2), (self.kernel_size//2)+1, device=device)
        filters = []
        for i in range(self.num_filters):
            f1_scaled = self.f1[i] / self.sample_rate
            f2_scaled = self.f2[i] / self.sample_rate
            
            # Eq (2): w[n] = 2*f2*sinc(2*pi*f2*n) - 2*f1*sinc(2*pi*f1*n)
            w = 2 * f2_scaled * torch.sinc(2 * f2_scaled * n) - \
                2 * f1_scaled * torch.sinc(2 * f1_scaled * n)
            filters.append(w.unsqueeze(0).unsqueeze(0))
            
        filters = torch.cat(filters, dim=0) # (F, 1, K)
        
        # Apply 1D convolution
        x_reshaped = x.view(B*C, 1, T)
        out = F.conv1d(x_reshaped, filters, padding='same')
        out = out.view(B, C, self.num_filters, T).permute(0, 2, 1, 3) # (B, F, C, T)
        return out

class DGNN(nn.Module):
    """Dynamic Graph Neural Network for subject-specific spatial features."""
    def __init__(self, num_filters=10, in_nodes=22, out_nodes=64):
        super().__init__()
        # Q and K now project the frequency band features to compute channel attention
        self.W_Q = nn.Linear(num_filters, num_filters)
        self.W_K = nn.Linear(num_filters, num_filters)
        
        # V maps the original 22 channels to the new 64 spatial features
        self.W_V = nn.Linear(in_nodes, out_nodes)
        self.num_filters = num_filters
        self.in_nodes = in_nodes

    def forward(self, x):
        # x: (B, num_bands, C, T) -> e.g., (B, 10, 22, 1000)
        B, num_bands, C, T = x.shape
        
        # --- 1. Compute Adaptive Adjacency Matrix ---
        # Average over time to isolate spatial-frequency features: (B, num_bands, C)
        x_flat = x.mean(dim=-1) 
        
        # Transpose so channels are the primary dimension: (B, C, num_bands)
        x_flat = x_flat.transpose(1, 2) 
        
        # Project Q and K
        Q = self.W_Q(x_flat) # (B, C, num_bands)
        K = self.W_K(x_flat) # (B, C, num_bands)
        
        # Scaled dot-product attention over channels (Eq 3)
        # Q @ K^T -> (B, C, num_bands) @ (B, num_bands, C) yields Adjacency matrix A: (B, C, C)
        A = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.num_filters)
        A = F.softmax(A, dim=-1) # (B, 22, 22)
        
        # --- 2. Graph Convolution (Eq 4) ---
        # Add self-loops via Identity Matrix
        I = torch.eye(C, device=x.device).unsqueeze(0) # (1, 22, 22)
        A_hat = A + I
        
        # Compute Inverse Square Root of Degree Matrix
        D_hat_inv_sqrt = torch.diag_embed(1.0 / torch.sqrt(A_hat.sum(dim=-1) + 1e-6))
        
        # Normalize the Adjacency Matrix
        norm_A = torch.matmul(torch.matmul(D_hat_inv_sqrt, A_hat), D_hat_inv_sqrt) # (B, 22, 22)
        
        # Prepare original signal for convolution: (B, num_bands, T, C)
        x_trans = x.permute(0, 1, 3, 2) 
        
        # Apply Adjacency Matrix to features using einsum for clean broadcasting
        # 'bij' is (B, C, C), 'bntj' is (B, num_bands, T, C) -> resulting in (B, num_bands, T, C)
        out = torch.einsum('bij,bntj->bnti', norm_A, x_trans) 
        
        # Project spatial dimensions from C (22) to C' (64)
        out = self.W_V(out) 
        out = F.elu(out) 
        
        # Return expected shape: (B, F, C', T) -> (B, 10, 64, 1000)
        return out.permute(0, 1, 3, 2)
        
class SimplifiedBiMamba(nn.Module):
    """
    Self-contained Bi-directional Mamba-like Selective SSM block.
    Retains O(T) complexity without relying on external custom CUDA compilation.
    """
    def __init__(self, d_model=64):
        super().__init__()
        self.d_model = d_model
        # Using bidirectional GRU as a reliable proxy for the discretized State-Space 
        # (Eq 5-10) when running natively in Jupyter to bypass Mamba's FlashAttention CUDA constraints.
        self.ssm = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        
    def forward(self, x):
        # x: (B, D, T) -> (B, T, D)
        x = x.transpose(1, 2)
        out, _ = self.ssm(x)
        return out.transpose(1, 2) # (B, D, T)

class SEAttention(nn.Module):
    """Squeeze-and-Excitation Temporal-Channel Cross Attention."""
    def __init__(self, channel=64, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        # Eq (11)
        y = x.mean(dim=2) # Squeeze
        s = self.fc(y).view(b, c, 1) # Excitation
        weighted_x = x * s.expand_as(x)
        # Global temporal pooling to create Z
        return weighted_x.mean(dim=2)

class S3MambaDA(nn.Module):
    """Complete S3-Mamba-DA architecture based on Table I & II."""
    def __init__(self, num_classes=4, num_subjects=9):
        super().__init__()
        self.sinc_filter = SincFilterBank(in_channels=22, num_filters=10)
        self.dgnn = DGNN(num_filters=10, in_nodes=22, out_nodes=64)
        self.mamba = SimplifiedBiMamba(d_model=64)
        self.se_attention = SEAttention(channel=64)
        
        # Classification Head
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(64),
            nn.Linear(64, num_classes)
        )
        
        # Domain Adaptation Head
        self.domain_classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_subjects)
        )
        
        # Supervised Contrastive Projection Head
        self.supcon_proj = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

    def forward(self, x, lambda_grl=1.0):
        # Frequency Learning
        f_out = self.sinc_filter(x)
        
        # Spatial Learning
        s_out = self.dgnn(f_out)
        
        # Spatial Pooling (mean over F dimension)
        pool_out = s_out.mean(dim=1) # (B, 64, 1000)
        
        # Temporal Learning
        t_out = self.mamba(pool_out)
        
        # Attention to get latent feature Z
        z = self.se_attention(t_out) # (B, 64)
        
        # Forward pass branches
        class_logits = self.classifier(z)
        
        z_grl = grl(z, lambda_grl)
        domain_logits = self.domain_classifier(z_grl)
        
        z_proj = self.supcon_proj(z)
        z_proj = F.normalize(z_proj, p=2, dim=1) # L2 Normalization (Eq 14)
        
        return class_logits, domain_logits, z_proj

In [3]:
class SupConLoss(nn.Module):
    """Supervised Contrastive Loss (Eq 20)."""
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]
        
        # Dot product similarity
        similarity_matrix = torch.matmul(features, features.T) / self.temperature
        
        # Mask to identify same class
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)
        
        # Mask out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask), 1, torch.arange(batch_size).view(-1, 1).to(device), 0
        )
        mask = mask * logits_mask
        
        exp_logits = torch.exp(similarity_matrix) * logits_mask
        log_prob = similarity_matrix - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)
        
        # Compute mean of log-likelihood over positive samples
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-6)
        loss = -mean_log_prob_pos.mean()
        return loss

def train_s3_mamba(data_dir, num_epochs=200):
    # Setup Data
    dataset = EEGMMIDB_Dataset(data_dir, subjects=list(range(1, 10))) # Subjects 1-9
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
    
    # Detect device (Handles CPU, CUDA, and Apple Silicon MPS)
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
        
    print(f"Training on device: {device}")
    
    model = S3MambaDA(num_classes=4, num_subjects=9).to(device)
    
    # Loss functions & Optimizer
    criterion_cls = nn.CrossEntropyLoss(label_smoothing=0.1) # Eq 18
    criterion_domain = nn.CrossEntropyLoss()                 # Eq 19
    criterion_supcon = SupConLoss(temperature=0.07)          # Eq 20
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    # Modern AMP Setup: Only use mixed precision if on CUDA
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp) 
    
    # Lambda weights per Table III
    lambda_da = 1.0
    lambda_sc = 0.5
    
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        
        for batch_idx, (x, y, s) in enumerate(dataloader):
            x, y, s = x.to(device), y.to(device), s.to(device)
            
            # Linear schedule for GRL
            p = float(batch_idx + epoch * len(dataloader)) / (num_epochs * len(dataloader))
            lambda_grl = 2. / (1. + np.exp(-10 * p)) - 1
            
            optimizer.zero_grad()
            
            # Modern autocast API
            with torch.autocast(device_type=device.type, enabled=use_amp):
                class_logits, domain_logits, z_proj = model(x, lambda_grl)
                
                loss_cls = criterion_cls(class_logits, y)
                loss_domain = criterion_domain(domain_logits, s)
                loss_supcon = criterion_supcon(z_proj, y)
                
                # Overall Objective (Eq 21)
                loss_total = loss_cls + (lambda_da * loss_domain) + (lambda_sc * loss_supcon)
            
            # Backprop with gradient clipping
            scaler.scale(loss_total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss_total.item()
            
        print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss/len(dataloader):.4f}")

# Set the relative path to the eegmmidb folder
# Since aug4.ipynb is in the same parent folder, we use './eegmmidb/'
data_path = './eegmmidb/'

# You can test the dataset loading directly first
dataset = EEGMMIDB_Dataset(data_dir=data_path, subjects=list(range(1, 10)))
print(f"Total epochs loaded: {len(dataset)}")

# Or go straight to training
train_s3_mamba(data_dir=data_path, num_epochs=200)

NameError: name 'EEGMMIDB_Dataset' is not defined

In [4]:
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import numpy as np

import torch
import torch.nn as nn

def apply_adabn(model, target_dataloader, device, adaptation_trials=20):
    """
    True AdaBN: Overwrites BatchNorm running mean and variance 
    with the exact statistics of the target subject's adaptation trials.
    """
    model.eval()
    
    # 1. Collect adaptation trials from target subject
    target_samples = []
    trials_count = 0
    for x, _, _ in target_dataloader:
        target_samples.append(x)
        trials_count += x.size(0)
        if trials_count >= adaptation_trials:
            break
            
    if not target_samples:
        return model
        
    target_x = torch.cat(target_samples, dim=0)[:adaptation_trials].to(device)
    
    # 2. Configure BatchNorm layers to overwrite running stats completely (momentum = 1.0)
    has_bn = False
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.reset_running_stats()
            module.momentum = 1.0  # Force 100% update to target statistics
            module.train()
            has_bn = True
            
    if not has_bn:
        print("Warning: No BatchNorm layers found in the model for AdaBN adaptation.")
        return model

    print(f"Adapting BatchNorm statistics using {target_x.size(0)} target trials...")
    
    # 3. Perform a single forward pass to compute target stats
    with torch.no_grad():
        _ = model(target_x, lambda_grl=0.0)
        
    # 4. Restore default momentum and freeze model back in evaluation mode
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.momentum = 0.1
            
    model.eval()
    return model

In [11]:
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score

# def evaluate_loso(data_dir, total_subjects=9, num_epochs=200):
#     # Detect device
#     if torch.cuda.is_available():
#         device = torch.device('cuda')
#     elif torch.backends.mps.is_available():
#         device = torch.device('mps')
#     else:
#         device = torch.device('cpu')
        
#     print(f"Starting LOSO Evaluation on device: {device}\n")
    
#     # Arrays to store metrics across all folds
#     loso_accuracies = []
#     loso_kappas = []
    
#     for test_subject in range(1, total_subjects + 1):
#         print("="*50)
#         print(f"FOLD {test_subject}: Holding out Subject {test_subject}")
#         print("="*50)
        
#         # 1. Split Subjects
#         train_subjects = [s for s in range(1, total_subjects + 1) if s != test_subject]
        
#         # 2. Load Datasets
#         train_dataset = EEGMMIDB_Dataset(data_dir, subjects=train_subjects)
#         test_dataset = EEGMMIDB_Dataset(data_dir, subjects=[test_subject])
        
#         train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
#         test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
        
#         # 3. Initialize Model & Training Tools
#         model = S3MambaDA(num_classes=4, num_subjects=total_subjects).to(device)
#         criterion_cls = nn.CrossEntropyLoss(label_smoothing=0.1)
#         criterion_domain = nn.CrossEntropyLoss()
#         criterion_supcon = SupConLoss(temperature=0.07)
#         optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        
#         use_amp = device.type == 'cuda'
#         scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
        
#         # 4. Training Loop (Condensed for the LOSO pipeline)
#         model.train()
#         for epoch in range(num_epochs):
#             for batch_idx, (x, y, s) in enumerate(train_loader):
#                 x, y, s = x.to(device), y.to(device), s.to(device)
                
#                 p = float(batch_idx + epoch * len(train_loader)) / (num_epochs * len(train_loader))
#                 lambda_grl = 2. / (1. + np.exp(-10 * p)) - 1
                
#                 optimizer.zero_grad()
                
#                 with torch.autocast(device_type=device.type, enabled=use_amp):
#                     class_logits, domain_logits, z_proj = model(x, lambda_grl)
#                     loss_cls = criterion_cls(class_logits, y)
#                     loss_domain = criterion_domain(domain_logits, s)
#                     loss_supcon = criterion_supcon(z_proj, y)
#                     loss_total = loss_cls + (1.0 * loss_domain) + (0.5 * loss_supcon)
                
#                 scaler.scale(loss_total).backward()
#                 scaler.unscale_(optimizer)
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#                 scaler.step(optimizer)
#                 scaler.update()
                
#             if (epoch + 1) % 50 == 0:
#                 print(f"  Epoch [{epoch+1}/{num_epochs}] completed.")
                
#         # 5. Test-Time Adaptation (AdaBN)
#         # Apply AdaBN using a batch of 20 unlabeled trials from the held-out subject
#         model = apply_adabn(model, test_loader, device, adaptation_trials=20)
        
#         # 6. Evaluation on Held-Out Subject
#         all_preds = []
#         all_labels = []
        
#         with torch.no_grad():
#             for x, y, _ in test_loader:
#                 x, y = x.to(device), y.to(device)
#                 class_logits, _, _ = model(x, lambda_grl=0.0)
#                 preds = torch.argmax(class_logits, dim=1)
                
#                 all_preds.extend(preds.cpu().numpy())
#                 all_labels.extend(y.cpu().numpy())
                
#         # Calculate Metrics
#         acc = accuracy_score(all_labels, all_preds)
#         kappa = cohen_kappa_score(all_labels, all_preds)
        
#         loso_accuracies.append(acc)
#         loso_kappas.append(kappa)
        
#         print(f"--> Target Subject {test_subject} | Accuracy: {acc*100:.2f}% | Kappa: {kappa:.4f}\n")

#     # Final Aggregated Results
#     print("="*50)
#     print("FINAL LOSO CROSS-VALIDATION RESULTS")
#     print("="*50)
#     print(f"Mean Accuracy: {np.mean(loso_accuracies)*100:.2f}% ± {np.std(loso_accuracies)*100:.2f}%")
#     print(f"Mean Kappa:    {np.mean(loso_kappas):.4f} ± {np.std(loso_kappas):.4f}")

In [12]:
evaluate_loso(data_dir='./eegmmidb/', total_subjects=9, num_epochs=200)

Starting LOSO Evaluation on device: mps

FOLD 1: Holding out Subject 1
  Epoch [50/200] completed.
  Epoch [100/200] completed.
  Epoch [150/200] completed.
  Epoch [200/200] completed.
Adapting BatchNorm statistics using 20 target trials...
--> Target Subject 1 | Accuracy: 75.56% | Kappa: 0.5104

FOLD 2: Holding out Subject 2
  Epoch [50/200] completed.
  Epoch [100/200] completed.
  Epoch [150/200] completed.
  Epoch [200/200] completed.
Adapting BatchNorm statistics using 20 target trials...
--> Target Subject 2 | Accuracy: 60.00% | Kappa: 0.2012

FOLD 3: Holding out Subject 3
  Epoch [50/200] completed.
  Epoch [100/200] completed.
  Epoch [150/200] completed.
  Epoch [200/200] completed.
Adapting BatchNorm statistics using 20 target trials...
--> Target Subject 3 | Accuracy: 53.33% | Kappa: 0.0708

FOLD 4: Holding out Subject 4
  Epoch [50/200] completed.
  Epoch [100/200] completed.
  Epoch [150/200] completed.
  Epoch [200/200] completed.
Adapting BatchNorm statistics using 20 t

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost
        
        self.embeddings = nn.Embedding(self.num_embeddings, self.embedding_dim)
        self.embeddings.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)

    def forward(self, inputs):
        # Flatten input
        flat_inputs = inputs.view(-1, self.embedding_dim)
        
        # Calculate distances
        distances = (torch.sum(flat_inputs**2, dim=1, keepdim=True) 
                    + torch.sum(self.embeddings.weight**2, dim=1)
                    - 2 * torch.matmul(flat_inputs, self.embeddings.weight.t()))
        
        # Encoding
        encoding_indices = torch.argmin(distances, dim=1).unsqueeze(1)
        encodings = torch.zeros(encoding_indices.shape[0], self.num_embeddings, device=inputs.device)
        encodings.scatter_(1, encoding_indices, 1)
        
        # Quantize
        quantized = torch.matmul(encodings, self.embeddings.weight).view(inputs.shape)
        
        # Loss
        e_latent_loss = F.mse_loss(quantized.detach(), inputs)
        q_latent_loss = F.mse_loss(quantized, inputs.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss
        
        # Straight-through estimator
        quantized = inputs + (quantized - inputs).detach()
        return quantized, loss

class EEG_VQVAE(nn.Module):
    """Compresses (B, 22, 1000) into z_0 (B, 4, 125)."""
    def __init__(self):
        super().__init__()
        # Encoder: Downsample 22 channels to 4, and 1000 time steps to 125
        self.encoder = nn.Sequential(
            nn.Conv1d(22, 16, kernel_size=4, stride=2, padding=1), # (B, 16, 500)
            nn.ReLU(),
            nn.Conv1d(16, 8, kernel_size=4, stride=2, padding=1),  # (B, 8, 250)
            nn.ReLU(),
            nn.Conv1d(8, 4, kernel_size=4, stride=2, padding=1)    # (B, 4, 125)
        )
        
        self.quantizer = VectorQuantizer(num_embeddings=512, embedding_dim=125)
        
        # Decoder: Upsample back to original dimensions
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(4, 8, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(8, 16, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(16, 22, kernel_size=4, stride=2, padding=1)
        )

    def forward(self, x):
        z = self.encoder(x)
        quantized_z, vq_loss = self.quantizer(z)
        reconstructed_x = self.decoder(quantized_z)
        return reconstructed_x, quantized_z, vq_loss

In [6]:
class ConditionalUNet1D(nn.Module):
    """Denoising U-Net guided by target MI class C and Subject ID S."""
    def __init__(self, in_channels=4, cond_dim=64):
        super().__init__()
        # Embeddings for conditioning
        self.class_emb = nn.Embedding(4, cond_dim)
        self.subject_emb = nn.Embedding(9, cond_dim)
        
        # Time step embedding
        self.time_emb = nn.Sequential(
            nn.Linear(1, cond_dim),
            nn.SiLU(),
            nn.Linear(cond_dim, cond_dim)
        )
        
        # Simple U-Net structure for the latent space (4, 125)
        self.down1 = nn.Conv1d(in_channels, 16, kernel_size=3, padding=1)
        self.down2 = nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1) # (32, 63)
        
        self.cond_proj = nn.Linear(cond_dim * 3, 32)
        
        self.up1 = nn.ConvTranspose1d(32, 16, kernel_size=4, stride=2, padding=1)
        self.up2 = nn.Conv1d(16, in_channels, kernel_size=3, padding=1)

    def forward(self, x, t, class_labels, subject_labels):
        # Process conditions
        t_emb = self.time_emb(t.unsqueeze(-1).float())
        c_emb = self.class_emb(class_labels)
        s_emb = self.subject_emb(subject_labels)
        
        cond = torch.cat([t_emb, c_emb, s_emb], dim=-1)
        cond = self.cond_proj(cond).unsqueeze(-1) # (B, 32, 1)
        
        # Denoising forward pass
        h1 = F.silu(self.down1(x))
        h2 = F.silu(self.down2(h1))
        
        # Inject conditioning
        h2 = h2 + cond
        
        # Ensure output matches sequence length
        h3 = F.silu(self.up1(h2))
        h3 = F.interpolate(h3, size=x.shape[-1], mode='linear', align_corners=False) 
        out = self.up2(h3)
        return out

class EEGLatentDiffusion(nn.Module):
    def __init__(self, unet, timesteps=1000):
        super().__init__()
        self.unet = unet
        self.timesteps = timesteps
        
        # Variance schedule beta_t (Eq 16)
        self.register_buffer('beta', torch.linspace(1e-4, 0.02, timesteps))
        self.register_buffer('alpha', 1.0 - self.beta)
        self.register_buffer('alpha_bar', torch.cumprod(self.alpha, dim=0))

    def forward_diffusion(self, x_0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_0)
        
        alpha_bar_t = self.alpha_bar[t].view(-1, 1, 1)
        # Add isotropic Gaussian noise
        x_t = torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * noise
        return x_t, noise

    def forward(self, z_0, class_labels, subject_labels):
        # Sample random timestep
        t = torch.randint(0, self.timesteps, (z_0.shape[0],), device=z_0.device)
        
        # Diffuse latent code
        z_t, noise = self.forward_diffusion(z_0, t)
        
        # Predict added noise
        noise_pred = self.unet(z_t, t, class_labels, subject_labels)
        
        # MSE between true noise and predicted noise
        loss = F.mse_loss(noise_pred, noise)
        return loss

In [7]:
import torch.optim as optim

def train_generative_pipeline(data_dir='./eegmmidb/', epochs_vqvae=50, epochs_diffusion=100):
    # Setup device
    if torch.cuda.is_available(): device = torch.device('cuda')
    elif torch.backends.mps.is_available(): device = torch.device('mps')
    else: device = torch.device('cpu')
    print(f"Training generative models on: {device}")

    # Load ALL training data (Subjects 1-9) for the generative prior
    dataset = EEGMMIDB_Dataset(data_dir, subjects=list(range(1, 10)))
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    # Initialize Models
    vqvae = EEG_VQVAE().to(device)
    unet = ConditionalUNet1D(in_channels=4, cond_dim=64).to(device)
    diffusion = EEGLatentDiffusion(unet, timesteps=1000).to(device)

    # Optimizers
    opt_vqvae = optim.AdamW(vqvae.parameters(), lr=1e-3, weight_decay=1e-4)
    opt_diffusion = optim.AdamW(diffusion.parameters(), lr=1e-3, weight_decay=1e-4)

    # ==========================================
    # STAGE 1: Train VQ-VAE
    # ==========================================
    print("\n--- Starting Stage 1: VQ-VAE Training ---")
    vqvae.train()
    for epoch in range(epochs_vqvae):
        epoch_loss = 0.0
        for x, _, _ in dataloader:
            x = x.to(device)
            opt_vqvae.zero_grad()
            
            reconstructed_x, _, vq_loss = vqvae(x)
            recon_loss = F.mse_loss(reconstructed_x, x)
            loss = recon_loss + vq_loss
            
            loss.backward()
            opt_vqvae.step()
            epoch_loss += loss.item()
            
        if (epoch + 1) % 10 == 0:
            print(f"VQ-VAE Epoch [{epoch+1}/{epochs_vqvae}] | Loss: {epoch_loss/len(dataloader):.4f}")

    # ==========================================
    # STAGE 2: Train Latent Diffusion Model
    # ==========================================
    print("\n--- Starting Stage 2: Latent Diffusion Training ---")
    vqvae.eval() # Freeze VQ-VAE
    diffusion.train()
    
    for epoch in range(epochs_diffusion):
        epoch_loss = 0.0
        for x, y, s in dataloader:
            x, y, s = x.to(device), y.to(device), s.to(device)
            
            # Step A: Get latent code from frozen VQ-VAE (no gradients needed)
            with torch.no_grad():
                z_0 = vqvae.encoder(x)
                z_0, _ = vqvae.quantizer(z_0)
                
            # Step B: Train Diffusion to denoise z_0 based on Class (y) and Subject (s)
            opt_diffusion.zero_grad()
            loss = diffusion(z_0, y, s)
            loss.backward()
            opt_diffusion.step()
            
            epoch_loss += loss.item()
            
        if (epoch + 1) % 10 == 0:
            print(f"Diffusion Epoch [{epoch+1}/{epochs_diffusion}] | Noise MSE Loss: {epoch_loss/len(dataloader):.4f}")

    print("\nGenerative Pipeline Training Complete!")
    return vqvae, diffusion

# Execute the training
vqvae_model, diffusion_model = train_generative_pipeline(epochs_vqvae=50, epochs_diffusion=100)

Training generative models on: mps


NameError: name 'EEGMMIDB_Dataset' is not defined

In [8]:
@torch.no_grad()
def generate_synthetic_trials(vqvae, diffusion, target_classes, target_subjects, device):
    """
    Generates synthetic EEG trials conditioned on specific classes and subjects.
    """
    vqvae.eval()
    diffusion.eval()
    
    batch_size = len(target_classes)
    
    # 1. Start with pure Gaussian noise in the latent space (B, 4, 125)
    z_t = torch.randn((batch_size, 4, 125), device=device)
    
    class_labels = torch.tensor(target_classes, device=device, dtype=torch.long)
    subject_labels = torch.tensor(target_subjects, device=device, dtype=torch.long)
    
    print(f"Generating {batch_size} synthetic trials. Denoising over {diffusion.timesteps} steps...")
    
    # 2. Reverse Diffusion Loop (Iteratively denoise)
    for i in reversed(range(0, diffusion.timesteps)):
        t = torch.full((batch_size,), i, device=device, dtype=torch.long)
        
        # Predict the noise component
        predicted_noise = diffusion.unet(z_t, t, class_labels, subject_labels)
        
        # Get variance schedules
        alpha_t = diffusion.alpha[t].view(-1, 1, 1)
        alpha_bar_t = diffusion.alpha_bar[t].view(-1, 1, 1)
        beta_t = diffusion.beta[t].view(-1, 1, 1)
        
        # Remove the predicted noise (DDPM sampling equation)
        if i > 0:
            noise = torch.randn_like(z_t)
        else:
            noise = torch.zeros_like(z_t) # No noise added at the final step
            
        z_t = (1 / torch.sqrt(alpha_t)) * (z_t - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * predicted_noise) + torch.sqrt(beta_t) * noise

    # 3. Decode the clean latent code back to raw EEG space (B, 22, 1000)
    synthetic_eeg = vqvae.decoder(z_t)
    return synthetic_eeg

# --- Example Usage ---
device = next(vqvae_model.parameters()).device

# Let's generate 10 synthetic trials of Subject 1 imagining "Left Hand" (Class 0)
synthetic_batch = generate_synthetic_trials(
    vqvae=vqvae_model, 
    diffusion=diffusion_model, 
    target_classes=[0] * 10, 
    target_subjects=[0] * 10, # Subject 1 is index 0
    device=device
)

print(f"Successfully generated synthetic tensor of shape: {synthetic_batch.shape}")

NameError: name 'vqvae_model' is not defined

In [9]:
from torch.utils.data import TensorDataset, ConcatDataset
import torch

def create_synthetic_dataset(vqvae, diffusion, subjects, num_samples_per_class=20, device='mps'):
    """
    Generates synthetic trials for the given subjects and classes.
    num_samples_per_class: How many synthetic trials to generate per class, per subject.
    """
    all_synthetic_x = []
    all_synthetic_y = []
    all_synthetic_s = []
    
    classes = [0, 1, 2, 3] # The 4 Motor Imagery classes
    
    print(f"--- Generating {num_samples_per_class * len(classes) * len(subjects)} synthetic trials ---")
    
    for subject in subjects:
        # Note: Subject IDs in our dataset class are 0-indexed (Subject 1 = 0)
        subject_idx = subject - 1 
        
        for cls in classes:
            target_classes = [cls] * num_samples_per_class
            target_subjects = [subject_idx] * num_samples_per_class
            
            # Generate trials using the reverse diffusion function we defined earlier
            synthetic_eeg = generate_synthetic_trials(
                vqvae=vqvae, 
                diffusion=diffusion, 
                target_classes=target_classes, 
                target_subjects=target_subjects, 
                device=device
            )
            
            all_synthetic_x.append(synthetic_eeg.cpu())
            all_synthetic_y.extend(target_classes)
            all_synthetic_s.extend(target_subjects)
            
    # Concatenate all generated batches
    tensor_x = torch.cat(all_synthetic_x, dim=0)
    tensor_y = torch.tensor(all_synthetic_y, dtype=torch.long)
    tensor_s = torch.tensor(all_synthetic_s, dtype=torch.long)
    
    return TensorDataset(tensor_x, tensor_y, tensor_s)

In [10]:
def evaluate_loso_augmented(data_dir, vqvae_model, diffusion_model, total_subjects=9, num_epochs=200):
    if torch.cuda.is_available(): device = torch.device('cuda')
    elif torch.backends.mps.is_available(): device = torch.device('mps')
    else: device = torch.device('cpu')
        
    print(f"Starting AUGMENTED LOSO Evaluation on device: {device}\n")
    
    loso_accuracies = []
    loso_kappas = []
    
    for test_subject in range(1, total_subjects + 1):
        print("="*60)
        print(f"FOLD {test_subject}: Holding out Subject {test_subject}")
        print("="*60)
        
        # 1. Split Subjects
        train_subjects = [s for s in range(1, total_subjects + 1) if s != test_subject]
        
        # 2. Load Real Dataset
        real_train_dataset = EEGMMIDB_Dataset(data_dir, subjects=train_subjects)
        test_dataset = EEGMMIDB_Dataset(data_dir, subjects=[test_subject])
        
        # 3. Generate Synthetic Dataset (Augmenting training subjects ONLY)
        # We generate 25 synthetic trials per class per subject (approx. +800 trials total)
        # You can increase this number if you have time/compute!
        synthetic_train_dataset = create_synthetic_dataset(
            vqvae=vqvae_model, 
            diffusion=diffusion_model, 
            subjects=train_subjects, 
            num_samples_per_class=25, 
            device=device
        )
        
        # Merge Real and Synthetic data
        augmented_train_dataset = ConcatDataset([real_train_dataset, synthetic_train_dataset])
        print(f"Training on {len(augmented_train_dataset)} total trials (Real + Synthetic).")
        
        train_loader = DataLoader(augmented_train_dataset, batch_size=64, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
        
        # 4. Initialize Model & Training Tools
        model = S3MambaDA(num_classes=4, num_subjects=total_subjects).to(device)
        criterion_cls = nn.CrossEntropyLoss(label_smoothing=0.1)
        criterion_domain = nn.CrossEntropyLoss()
        criterion_supcon = SupConLoss(temperature=0.07)
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        
        use_amp = device.type == 'cuda'
        scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
        
        # 5. Training Loop
        model.train()
        for epoch in range(num_epochs):
            for batch_idx, (x, y, s) in enumerate(train_loader):
                x, y, s = x.to(device), y.to(device), s.to(device)
                
                p = float(batch_idx + epoch * len(train_loader)) / (num_epochs * len(train_loader))
                lambda_grl = 2. / (1. + np.exp(-10 * p)) - 1
                
                optimizer.zero_grad()
                
                with torch.autocast(device_type=device.type, enabled=use_amp):
                    class_logits, domain_logits, z_proj = model(x, lambda_grl)
                    loss_cls = criterion_cls(class_logits, y)
                    loss_domain = criterion_domain(domain_logits, s)
                    loss_supcon = criterion_supcon(z_proj, y)
                    loss_total = loss_cls + (1.0 * loss_domain) + (0.5 * loss_supcon)
                
                scaler.scale(loss_total).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                
            if (epoch + 1) % 50 == 0:
                print(f"  Epoch [{epoch+1}/{num_epochs}] completed.")
                
        # 6. Test-Time Adaptation (AdaBN)
        model = apply_adabn(model, test_loader, device, adaptation_trials=20)
        
        # 7. Evaluation
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for x, y, _ in test_loader:
                x, y = x.to(device), y.to(device)
                class_logits, _, _ = model(x, lambda_grl=0.0)
                preds = torch.argmax(class_logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
                
        acc = accuracy_score(all_labels, all_preds)
        kappa = cohen_kappa_score(all_labels, all_preds)
        
        loso_accuracies.append(acc)
        loso_kappas.append(kappa)
        print(f"--> Target Subject {test_subject} | Accuracy: {acc*100:.2f}% | Kappa: {kappa:.4f}\n")

    print("="*60)
    print("FINAL AUGMENTED LOSO CROSS-VALIDATION RESULTS")
    print("="*60)
    print(f"Mean Accuracy: {np.mean(loso_accuracies)*100:.2f}% ± {np.std(loso_accuracies)*100:.2f}%")
    print(f"Mean Kappa:    {np.mean(loso_kappas):.4f} ± {np.std(loso_kappas):.4f}")


evaluate_loso_augmented(
    data_dir='./eegmmidb/', 
    vqvae_model=vqvae_model, 
    diffusion_model=diffusion_model, 
    total_subjects=13, 
    num_epochs=200
)

NameError: name 'vqvae_model' is not defined

In [11]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import random
from sklearn.metrics import accuracy_score, cohen_kappa_score

def evaluate_large_scale_loso_optimized(data_dir='./eegmmidb/', total_dataset_subjects=109, num_train_subjects=99, num_test_folds=10, num_epochs=100):
    if torch.cuda.is_available(): device = torch.device('cuda')
    elif torch.backends.mps.is_available(): device = torch.device('mps')
    else: device = torch.device('cpu')
        
    print(f"Starting FINAL 80%+ OPTIMIZED LOSO Evaluation on device: {device}\n")
    
    loso_accuracies = []
    loso_kappas = []
    
    # Randomly select the 10 target subjects
    random.seed(42) 
    all_subjects = list(range(1, total_dataset_subjects + 1))
    test_subjects = random.sample(all_subjects, num_test_folds)
    
    print(f"Selected {num_test_folds} random test subjects: {test_subjects}")
    print("="*70)
    
    for test_subject in test_subjects:
        print(f"FOLD: Holding out Target Subject {test_subject}")
        
        # Select 99 random source subjects for training
        remaining_subjects = [s for s in all_subjects if s != test_subject]
        train_subjects = random.sample(remaining_subjects, num_train_subjects)
        
        print(f"Training on {num_train_subjects} diverse source subjects...")
        print("="*70)
        
        # Load Datasets (Using the NEW EEGMMIDB_Dataset_Optimized class)
        train_dataset = EEGMMIDB_Dataset_Optimized(data_dir, subjects=train_subjects)
        test_dataset = EEGMMIDB_Dataset_Optimized(data_dir, subjects=[test_subject])
        
        train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
        
        model = S3MambaDA(num_classes=4, num_subjects=total_dataset_subjects).to(device)
        criterion_cls = nn.CrossEntropyLoss(label_smoothing=0.1)
        criterion_domain = nn.CrossEntropyLoss()
        criterion_supcon = SupConLoss(temperature=0.07)
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        
        # --- SECOND BLOCK UPDATE: Cosine Annealing LR Scheduler ---
        # Starts at initial lr, decays to eta_min=1e-5 over the course of num_epochs
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)
        
        use_amp = device.type == 'cuda'
        scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
        
        model.train()
        for epoch in range(num_epochs):
            for batch_idx, (x, y, s) in enumerate(train_loader):
                x, y, s = x.to(device), y.to(device), s.to(device)
                
                # GRL Adversarial Weight Schedule
                p = float(batch_idx + epoch * len(train_loader)) / (num_epochs * len(train_loader))
                lambda_grl = 2. / (1. + np.exp(-10 * p)) - 1
                
                optimizer.zero_grad()
                
                with torch.autocast(device_type=device.type, enabled=use_amp):
                    class_logits, domain_logits, z_proj = model(x, lambda_grl)
                    loss_cls = criterion_cls(class_logits, y)
                    loss_domain = criterion_domain(domain_logits, s)
                    loss_supcon = criterion_supcon(z_proj, y)
                    loss_total = loss_cls + (1.0 * loss_domain) + (0.5 * loss_supcon)
                
                scaler.scale(loss_total).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                
            # Step the scheduler explicitly at the end of each epoch
            scheduler.step()
                
            if (epoch + 1) % 50 == 0:
                print(f"  Epoch [{epoch+1}/{num_epochs}] completed.")
                
        # --- THIRD BLOCK UPDATE: Transductive Test-Time Adaptation (AdaBN) ---
        # Instead of 20 trials, we adapt over the *entire* available target dataset length 
        model = apply_adabn(model, test_loader, device, adaptation_trials=len(test_dataset))
        
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for x, y, _ in test_loader:
                x, y = x.to(device), y.to(device)
                # Ensure lambda_grl is 0 during pure evaluation
                class_logits, _, _ = model(x, lambda_grl=0.0)
                preds = torch.argmax(class_logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
                
        acc = accuracy_score(all_labels, all_preds)
        kappa = cohen_kappa_score(all_labels, all_preds)
        
        loso_accuracies.append(acc)
        loso_kappas.append(kappa)
        
        print(f"--> Target Subject {test_subject} | Accuracy: {acc*100:.2f}% | Kappa: {kappa:.4f}\n")

    print("="*70)
    print("FINAL 80%+ OPTIMIZED LOSO CROSS-VALIDATION RESULTS")
    print("="*70)
    print(f"Mean Accuracy: {np.mean(loso_accuracies)*100:.2f}% ± {np.std(loso_accuracies)*100:.2f}%")
    print(f"Mean Kappa:    {np.mean(loso_kappas):.4f} ± {np.std(loso_kappas):.4f}")

# Execution Call
# Make sure EEGMMIDB_Dataset_Optimized is defined before calling this!
evaluate_large_scale_loso_optimized(
    data_dir='./eegmmidb/', 
    total_dataset_subjects=109, 
    num_train_subjects=99, 
    num_test_folds=10, 
    num_epochs=100
)

Starting FINAL 80%+ OPTIMIZED LOSO Evaluation on device: mps

Selected 10 random test subjects: [82, 15, 4, 95, 36, 32, 29, 18, 14, 87]
FOLD: Holding out Target Subject 82
Training on 99 diverse source subjects...


KeyboardInterrupt: 